# Notebook 2 – ML Model Training & Evaluation

This notebook covers the complete ML pipeline for CodeSense's error classifier:
- Feature engineering (TF-IDF on combined code + error text)
- Logistic Regression training with class balancing
- Cross-validation and performance metrics
- Confusion matrix and per-class precision/recall
- Confidence calibration analysis

**Model:** Logistic Regression with TF-IDF vectorizer  
**Target:** 8 error categories from the CodeSense dataset

In [ ]:
import sys
sys.path.insert(0, '../backend')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
from sklearn.pipeline import Pipeline
from ml.preprocessing import clean_text, combine_features
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

In [ ]:
# ── Load and preprocess the dataset ──────────────────────────────────────────
df = pd.read_csv('../dataset/error_dataset.csv')

# Apply the same feature combination used in production training
df['features'] = df.apply(combine_features, axis=1)
df['features_clean'] = df['features'].apply(clean_text)

X = df['features_clean']
y = df['error_category']

print(f'Dataset loaded: {len(df)} rows')
print(f'Categories: {sorted(y.unique())}')
print(f'\nSample feature string:')
print(X.iloc[0][:200])

In [ ]:
# ── Train/test split ──────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'\nClass distribution in training set:')
print(y_train.value_counts().to_string())

In [ ]:
# ── Build and train the pipeline (mirrors production) ─────────────────────────
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=8000,
        sublinear_tf=True,
        min_df=1,
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        C=1.0,
        solver='lbfgs',
        multi_class='multinomial',
        random_state=42
    ))
])

pipeline.fit(X_train, y_train)
print('Model trained successfully.')

In [ ]:
# ── Test set evaluation ───────────────────────────────────────────────────────
y_pred = pipeline.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average='macro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

print(f'Accuracy          : {acc:.4f} ({acc*100:.2f}%)')
print(f'F1 Macro          : {f1_macro:.4f}')
print(f'F1 Weighted       : {f1_weighted:.4f}')
print()
print('Per-class Classification Report:')
print(classification_report(y_test, y_pred, digits=3))

In [ ]:
# ── Stratified 5-Fold Cross Validation ───────────────────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X, y, cv=skf, scoring='accuracy', n_jobs=-1)

print('5-Fold Cross Validation:')
for fold, score in enumerate(cv_scores, 1):
    print(f'  Fold {fold}: {score:.4f}')
print(f'\nMean CV Accuracy : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# ── Confusion matrix ─────────────────────────────────────────────────────────
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, data, title, fmt in [
    (axes[0], cm, 'Confusion Matrix (Raw Counts)', 'd'),
    (axes[1], cm_norm, 'Confusion Matrix (Normalised)', '.2f'),
]:
    im = ax.imshow(data, interpolation='nearest', cmap='Blues')
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels([l.replace(' Error','\nError') for l in labels], fontsize=8)
    ax.set_yticklabels([l.replace(' Error','\nError') for l in labels], fontsize=8)
    for i in range(len(labels)):
        for j in range(len(labels)):
            val = data[i, j]
            color = 'white' if (val > data.max()/2 and fmt == 'd') or (val > 0.5) else 'black'
            ax.text(j, i, format(val, fmt), ha='center', va='center', color=color, fontsize=7)
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    ax.set_title(title, fontweight='bold')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix.png')

In [ ]:
# ── Top TF-IDF features per class ─────────────────────────────────────────────
vectorizer = pipeline.named_steps['tfidf']
classifier = pipeline.named_steps['clf']
feature_names = vectorizer.get_feature_names_out()

print('Top 10 discriminating features per error category:')
for i, cls in enumerate(classifier.classes_):
    top_indices = np.argsort(classifier.coef_[i])[-10:][::-1]
    top_features = [feature_names[idx] for idx in top_indices]
    print(f'\n{cls}:')
    print('  ' + ', '.join(top_features))

In [ ]:
# ── Confidence calibration: max probability distribution ─────────────────────
proba = pipeline.predict_proba(X_test)
max_conf = proba.max(axis=1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(max_conf, bins=20, color='#3b82f6', edgecolor='white', alpha=0.85)
ax.axvline(max_conf.mean(), color='#ef4444', linestyle='--', linewidth=1.5,
           label=f'Mean = {max_conf.mean():.3f}')
ax.set_title('Model Confidence Distribution (max softmax probability)', fontweight='bold')
ax.set_xlabel('Confidence Score')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.savefig('confidence_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Mean confidence   : {max_conf.mean():.3f}')
print(f'Median confidence : {np.median(max_conf):.3f}')
print(f'Low confidence (<0.5): {(max_conf < 0.5).sum()} samples')

In [ ]:
# ── Live prediction demonstration ─────────────────────────────────────────────
test_cases = [
    ("if x > 5 print(x) SyntaxError colon missing", "Syntax Error"),
    ("print(undefined_var) NameError not defined",   "Name Error"),
    ("'hello' + 5 TypeError str int",               "Type Error"),
    ("items[9] IndexError list index out of range",  "Index Error"),
    ("import missing_pkg ModuleNotFoundError",        "Import Error"),
    ("    print(x) IndentationError unexpected",     "Indentation Error"),
    ("10 / 0 ZeroDivisionError division runtime",    "Runtime Error"),
    ("wrong condition AssertionError logical error", "Logical Error"),
]

print(f'{"Input Text":50s}  {"Expected":18s}  {"Predicted":18s}  Conf   Match')
print('-' * 110)
correct = 0
for text, expected in test_cases:
    pred = pipeline.predict([text])[0]
    conf = pipeline.predict_proba([text]).max()
    match = '✓' if pred == expected else '✗'
    if pred == expected:
        correct += 1
    print(f'{text[:48]:50s}  {expected:18s}  {pred:18s}  {conf:.2f}   {match}')

print(f'\nDemo accuracy: {correct}/{len(test_cases)} ({correct/len(test_cases)*100:.0f}%)')